In [209]:
from langchain.memory import ConversationBufferWindowMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate

# 메모리 사용 LLM 영화 이모티콘 추출기
class MemoryLlmMovieticon:
    MEMORY_KEY = 'history'                  # 메모리와 관련된 템플릿 키 값 

    def __init__(self):
        # 모델, 메모리, 프롬프트 설정
        self.llm = self.getLLM()
        self.memory = self.getMemory(llm = self.llm)
        self.prompt = self.getPrompt()

        # 체인 설정
        self.chain = self.setChain(
            llm = self.llm,
            prompt = self.prompt,
        )
    
    """
      모델 설정
    """
    def getLLM(self):
        return ChatOpenAI(
            # model_name = 'gpt-4o-mini',
            temperature = 0.1
        )
    
    """
      메모리 설정
    """
    def getMemory(self, llm):
        # llm 사용
        # return ConversationSummaryBufferMemory(
        #     llm = llm,                      # 모델
        #     max_token_limit = 200,          # 최근 대화내역 최대 토큰 갯수
        #     return_messages = True,         # 대화 내역을 문자열 → message Class 사용
        #     memory_key = self.MEMORY_KEY,   # 기본 값이 'history'이기 때문에 self.MEMORY_KEY가 변하지 않는다면 사용하지 않아도 됨
        # )

        # llm 미사용
        return ConversationBufferWindowMemory(
            return_messages = True,         # 대화 내역을 문자열 → message Class 사용
            memory_key = self.MEMORY_KEY,   # 기본 값이 'history'이기 때문에 self.MEMORY_KEY가 변하지 않는다면 사용하지 않아도 됨
            k = 10,                         # 최근 대화 내역 몇 개까지 기록 할 건지 
        )
    
    """
      프롬프트 템플릿 설정
    """
    def getPrompt(self):
        return ChatPromptTemplate.from_messages([
            ("system",
                "당신은 사람과 AI와의 대화를 도와주는 역할과 영화 전문가를 담당하고 있습니다. "+ \
                "당신은 영화에 상징하는 내용을 이모티콘으로 답변합니다. "+ \
                "이모티콘은 꼭 3개로 추려서 답변합니다. "
            ),                                                              # 시스템 기술 설정
            self.getExamPrompt(),                                           # 영화 질문 응답 가이드 프롬프트
            MessagesPlaceholder(variable_name = self.MEMORY_KEY),           # "{history}"와 같은 템플릿으로 메모리에서 설정된 대화 내역을 교체
            ("human", "{question}"),                                        # 질문
        ])
    
    """
      응답결과 가이드
    """
    def getExamGuides(self):
        return [{
            "movie": "탑건",
            "answer": "🛩️👨‍✈️🔥",
        }, {
            "movie": "대부",
            "answer": "👨‍👨‍👦🔫🍝",
        }, {
            "movie": "기생충",
            "answer": "🏠🐜💰",
        }, {
            "movie": "오징어게임",
            "answer": "🦑💰🎮",
        }, ]
    
    """
      가이드 프롬프트 가져오기
    """
    def getExamPrompt(self):
        # 가이드 프롬프트
        exam_prompt = ChatPromptTemplate.from_messages([
            ('human', "{movie}"),
            ('ai', "{answer}")
        ])

        # 위 가이드 프롬프트 템플릿 적용
        return FewShotChatMessagePromptTemplate(
            example_prompt = exam_prompt,
            examples = self.getExamGuides()
        )
    
    """
      메모리 가져오기
    """
    def load_memory(self, _):
        return self.memory.load_memory_variables({})[self.MEMORY_KEY]       # 메모리에 기록 된 내역 가져오기
    
    """
      체인 설정
    """
    def setChain(self, prompt, llm):
        # 체인 설정 (메모리 대화내역 | 프롬프트 | 모델)
        return (
            RunnablePassthrough.assign(history=self.load_memory) # prompt의 MessagesPlaceholder에 전달 할 메모리 값 설정
            | prompt                                             # 프롬프트 처리
            | llm                                                # llm 연결
        )


    """
      체인 실행
    """
    def chain_invoke(self, question):
        # 체인 실행
        result = self.chain.invoke({"question": question})
        
        # 메모리에 기록
        self.memory.save_context({"input": question}, {"output": result.content})
        
        return result
    
    """
      결과 가져오기
    """
    def run(self, movie):
        res = self.chain_invoke(movie)
        return res.content

# 클래스 생성
chat = MemoryLlmMovieticon()

# 영화이름 설정 
chat.run("기생충")

# 이모티콘이 아닌 \u200d 같은 코드가 출력 된다면,
# 이는 이모티콘 결합에 사용되는 유니코드 조합을 렌더링 하지 못하는 상황
# 지원되는 정상적인 환경에서 출력했을 시 조합된 이모티콘으로 표현됩니다.

'🏠🐜💰'

In [210]:
chat.run("드래곤볼")

'🐉💥🥋'

In [211]:
chat.run("범죄도시3")

'🔫🚓💰'

In [212]:
chat.load_memory('')

[HumanMessage(content='기생충'),
 AIMessage(content='🏠🐜💰'),
 HumanMessage(content='드래곤볼'),
 AIMessage(content='🐉💥🥋'),
 HumanMessage(content='범죄도시3'),
 AIMessage(content='🔫🚓💰')]